Name: Sydney Thompson

Finding + upvoting 3-4 notebooks/discussion posts that you find particularly insightful or could realistically apply in your own modeling. Provide the URLs 

1. https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687104

Title: Effective number of features

I upvoted this notebook/discussion post because I wanted to see what others thought were the most important features. It's insightful for me because when I do my own analyses, I want to see how they compare to what I find.


2. https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687069

Title: Caution, the public leaderboard only has 1800 High Class

I upvoted this notebook/discussion because I thought it was important to note that there is a high overfitting risk. With the number of high class samples being 1801, that is a very low percentage. It is insightful because it will help me understand why it may be harder to tune/improve my models. 

3. https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687082

Title: Remember to tune your model thresholds at the end to enjoy a boost in scores

I upvoted this notebook/discussion because it's a good reminder to not jump into hyper-tuning right away. It's insightful to not be discouraged and not worry too much about a boost in scores. These may only increase slightly or not at all.

Initial EDA URL: 
https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/initial_EDA.ipynb

Bagging model (Random Forest) URL:
https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/random_forest.ipynb

Boosting model URLs:

XGBoost: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/xgboost_model.ipynb

LightGBM: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/lightgbm_model.ipynb

CatBoost: https://github.com/sydneythompson11/Advanced-ML/blob/main/Kaggle_Playground_Series/playground-series-s6e4/catboost_model.ipynb

## Modeling Discussion

### What I tried

This week I extended the boosting work to include three gradient boosting algorithms: XGBoost, LightGBM, and CatBoost. Each notebook follows the same structure — load data, encode the target, split 80/20 stratified, explore 3 hyperparameter sets, then retrain the best set on the full training data and generate a Kaggle submission.

For all three algorithms I addressed the class imbalance (the 'High' class is rare) by using either `class_weight='balanced'` / `auto_class_weights='Balanced'` or `compute_sample_weight`. This was the single biggest improvement from the previous week.

### Hyperparameter exploration

For each algorithm I deliberately chose three sets that differ in meaningful ways:

**XGBoost** (GridSearchCV over 16 combinations, 3-fold CV):
- Varied `n_estimators` (100 vs 200), `max_depth` (3 vs 6), `learning_rate` (0.05 vs 0.1), `subsample` (0.8 vs 1.0)
- Best: n_estimators=200, max_depth=6, lr=0.1, subsample=0.8 → CV balanced accuracy 0.9685

**LightGBM** (manual grid, 3 sets):
- Set 1: 100 trees, lr=0.15, 31 leaves, no regularisation — fast but underfits
- Set 2: 300 trees, lr=0.05, 127 leaves, L1+L2 reg=1.0 — better generalisation
- Set 3: 500 trees, lr=0.02, 255 leaves, reg=3.0, min_child_samples=50, colsample=0.8 — strong regularisation
- The jump from Set 1 to Set 2 was the most meaningful improvement; Set 3 was similar to Set 2 or slightly lower due to over-regularisation.

**CatBoost** (manual grid, 3 sets):
- Set 1: 100 iterations, lr=0.15, depth=4, l2=1 — shallow and fast, noticeably lower score
- Set 2: 300 iterations, lr=0.07, depth=7, l2=5, border_count=64 — solid middle ground
- Set 3: 500 iterations, lr=0.03, depth=10, l2=10, random_strength=3.0 — deeper trees with heavy regularisation
- CatBoost's native categorical handling (no manual encoding needed) was a clear advantage and likely explains why it matched or beat XGBoost with less preprocessing.

### What worked well

- Handling class imbalance was the most impactful change across all models. Without it, recall on the 'High' class was poor.
- CatBoost's `auto_class_weights='Balanced'` and native categorical support made it the easiest to set up and produced competitive scores.
- Increasing `n_estimators` / `iterations` with a lower `learning_rate` consistently helped, as long as regularisation was also increased to prevent overfitting.

### What didn't work well

- Very deep trees (depth=10 in CatBoost, 255 leaves in LightGBM) with insufficient regularisation tended to overfit on the training split without improving validation scores.
- GridSearchCV for XGBoost took 20-25 minutes for only 16 combinations × 3 folds. Scaling this up is expensive.
- The improvements between hyperparameter sets were generally small (0.001–0.01 balanced accuracy), which aligns with the competition discussion noting that the public leaderboard has very few 'High' class samples and scores are tightly clustered.

### Were improvements meaningful?

Mostly small. The biggest single gain came from adding class-weight handling (roughly +0.02–0.03 balanced accuracy). After that, hyperparameter tuning produced incremental gains of 0.001–0.008. This is consistent with the competition discussion warning about overfitting to the public leaderboard.

## Performance Metrics & Leaderboard Scores

| Model Type | Algorithm  | Hyperparameter Set | Val Balanced Accuracy | Kaggle LB Score |
|------------|------------|--------------------|-----------------------|-----------------|
| Bagging    | Random Forest | GridSearchCV best | 0.9616 | 0.95986 |
| Boosting   | XGBoost    | Set 1: n_est=100, depth=3, lr=0.05, sub=0.8 | 0.962 | — |
| Boosting   | XGBoost    | Set 2: n_est=100, depth=6, lr=0.1, sub=1.0  | 0.965 | — |
| Boosting   | XGBoost    | Best (GridSearchCV): n_est=200, depth=6, lr=0.1, sub=0.8 | 0.9685 | 0.95944 |
| Boosting   | LightGBM   | Set 1: 100 trees, lr=0.15, 31 leaves, no reg | 0.960 | — |
| Boosting   | LightGBM   | Set 2: 300 trees, lr=0.05, 127 leaves, reg=1.0 | 0.968 | — |
| Boosting   | LightGBM   | Set 3: 500 trees, lr=0.02, 255 leaves, reg=3.0 | 0.967 | 0.96833 |
| Boosting   | CatBoost   | Set 1: 100 iter, lr=0.15, depth=4, l2=1 | 0.955 | — |
| Boosting   | CatBoost   | Set 2: 300 iter, lr=0.07, depth=7, l2=5 | 0.970 | — |
| Boosting   | CatBoost   | Set 3: 500 iter, lr=0.03, depth=10, l2=10 | 0.971 | 0.96309 |

## How did the different boosting models compare?

All three boosting algorithms performed similarly on validation balanced accuracy (roughly 0.96–0.97), which is consistent with the competition being a relatively clean tabular dataset.

XGBoost required the most setup — manual label encoding for categoricals and explicit sample weights — but GridSearchCV made hyperparameter search systematic. It achieved 0.9685 CV balanced accuracy.

LightGBM was faster to train than XGBoost at equivalent tree counts, and the `num_leaves` parameter gave finer control over model complexity than `max_depth` alone. The jump from 31 to 127 leaves was the most impactful single change.

CatBoost was the easiest to work with because it handles categorical columns natively (no encoding step) and `auto_class_weights='Balanced'` is a single parameter. It also produced the highest validation scores in this experiment, likely because the dataset has categorical features that CatBoost's ordered target statistics handle particularly well.

The Kaggle leaderboard scores were tighter than the validation scores suggest — Random Forest (0.95986) actually edged out XGBoost (0.95944), which is a good reminder that validation score and leaderboard score don't always rank models the same way, especially with a small minority class.

## Phase 2 Plan

I plan on using Optuna next. After learning about it in class, I think it will help me find better parameters more efficiently than a manual grid. Additionally, I think it will run faster on my machine.

Feature engineering would also be interesting — looking into ratios between `Temperature_C` and `Soil_Moisture` may give better signals for the minority class.

I think it would also be interesting to look into ensemble/stacking models. Combining the predictions from Random Forest, XGBoost, LightGBM, and CatBoost via a simple vote or a meta-learner could push the leaderboard score higher than any single model.